In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-11'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0376, -0.2208, -0.4362, -0.0409, -0.8625, -0.0691, -0.1076,  0.3695,
          0.1815,  0.1230, -1.0229,  0.3485]], device='cuda:0')
Scaled actions :  tensor([[-0.0376, -0.2208, -0.4362, -0.0409, -0.8625, -0.0691, -0.1076,  0.3695,
          0.1815,  0.1230, -1.0229,  0.3485]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -3.7575e-02, -2.2083e-01,
         -4.3623e-01, -4.0945e-02, -8.6254e-01, -6.9059e-02, -1.0755e-01,
          3.6948e-01,  1.8147e-01,  1.2302e-01, -1.0229e+00,  3.4854e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.0914, -0.7070, -0.4227,  0.3990, -1.4332, -0.1366,  0.0761,  0.3619,
          0.5083,  0.5330, -1.3888,  0.6529]], device='cuda:0')
Scaled actions :  tensor([[-0.0914, -0.7070, -0.4227,  0.3990, -1.4332, -0.1366,  0.0761,  0.3619,
          0.5083,  0.5330, -1.3888,  0.6529]], device='cuda:0')
obs :  tensor([[ 4.3138e-02, -8.2127e-02,  5.6358e-01, -2.1463e-03, -1.0055e-03,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.6703e-02,
         -4.0326e-03, -8.0266e-03,  2.1714e-02, -8.9866e-02, -2.5810e-02,
         -4.2166e-02,  1.4331e-03,  1.3688e-03,  2.8874e-02, -1.0511e-01,
          7.5545e-02, -1.3941e-01, -3.8514e-02, -6.4894e-02,  1.7066e-01,
         -8.1222e-01, -2.1952e-01, -3.7029e-01,  1.5298e-02,  2.5424e-02,
          2.2843e-01, -9.4996e-01,  6.9001e-01, -9.1439e-02, -7.0701e-01,
         -4.2274e-01,  3.9904e-01, -1.4332e+00, -1.3657e-01,  7.6126e-02,
          3.6191e-01,  5.0828e-01,  5.3304e-01, -1.3888e+00,  6.5

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.5251,  0.0175, -0.2061, -1.3701,  0.0892,  0.3297,  0.9662,  0.4205,
          0.2533, -0.8589,  0.3075,  0.4161]], device='cuda:0')
Scaled actions :  tensor([[ 0.5251,  0.0175, -0.2061, -1.3701,  0.0892,  0.3297,  0.9662,  0.4205,
          0.2533, -0.8589,  0.3075,  0.4161]], device='cuda:0')
obs :  tensor([[-6.9752e-02, -1.2973e-01,  4.4548e-01, -6.5973e-03, -1.1013e-04,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.0875e-02,
         -1.5320e-02, -3.8946e-02,  9.0010e-02, -3.4235e-01, -7.4613e-02,
         -6.3539e-02,  9.9293e-03,  9.3232e-03,  9.0558e-02, -3.6422e-01,
          2.6042e-01, -7.7296e-02, -7.7628e-02, -2.2431e-01,  4.6880e-01,
         -1.5762e+00, -1.4804e-01,  1.0269e-01,  7.8739e-02,  5.2490e-02,
          3.6357e-01, -1.5462e+00,  8.5284e-01,  5.2512e-01,  1.7474e-02,
         -2.0607e-01, -1.3701e+00,  8.9165e-02,  3.2969e-01,  9.6619e-01,
          4.2048e-01,  2.5327e-01, -8.5886e-01,  3.0747e-01,  4.1

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.1663, -0.5839, -0.1828, -1.4709,  0.5270,  0.0769, -0.5627, -1.1820,
          0.2095, -1.3209,  0.7817, -0.1930]], device='cuda:0')
Scaled actions :  tensor([[-0.1663, -0.5839, -0.1828, -1.4709,  0.5270,  0.0769, -0.5627, -1.1820,
          0.2095, -1.3209,  0.7817, -0.1930]], device='cuda:0')
obs :  tensor([[-6.0696e-01, -2.2893e-01, -4.4669e-02, -1.3808e-02,  1.5223e-02,
         -9.9979e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.8314e-03,
         -1.5361e-02, -7.4841e-02,  1.6499e-01, -5.5009e-01, -7.9876e-04,
          1.2668e-02,  5.5754e-02,  4.2020e-02,  1.2254e-01, -5.6870e-01,
          3.3854e-01,  4.6671e-01,  5.0558e-02, -1.4835e-01,  3.0255e-01,
         -5.9938e-01,  6.8232e-01,  6.0796e-01,  3.4792e-01,  2.3958e-01,
         -4.5606e-03, -5.9465e-01,  1.7715e-01, -1.6630e-01, -5.8385e-01,
         -1.8276e-01, -1.4709e+00,  5.2697e-01,  7.6928e-02, -5.6270e-01,
         -1.1820e+00,  2.0951e-01, -1.3209e+00,  7.8172e-01, -1.9

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.4687, -0.0059, -0.6604, -0.4950,  0.1075,  0.2484, -0.6758, -0.9583,
         -0.5317, -1.1179,  0.4170,  0.2349]], device='cuda:0')
Scaled actions :  tensor([[-0.4687, -0.0059, -0.6604, -0.4950,  0.1075,  0.2484, -0.6758, -0.9583,
         -0.5317, -1.1179,  0.4170,  0.2349]], device='cuda:0')
obs :  tensor([[ 0.0369, -0.1062,  0.6694, -0.0209,  0.0256, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0361, -0.0308, -0.1027,  0.1980, -0.5632,  0.0658,  0.0676,
          0.1021,  0.0989,  0.0988, -0.5817,  0.2682, -0.0841, -0.1812, -0.1365,
          0.0501,  0.3708,  0.0482,  0.0078,  0.1273,  0.3072, -0.1985,  0.3707,
         -0.7851, -0.4687, -0.0059, -0.6604, -0.4950,  0.1075,  0.2484, -0.6758,
         -0.9583, -0.5317, -1.1179,  0.4170,  0.2349]], device='cuda:0')
torques: [-200.         -200.          -29.83407281 -200.          200.
  -44.987002   -200.         -200.          -71.22147239 -200.
  200.         -200.        ]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.6352,  0.9954,  0.2346, -0.3792, -0.0067,  0.1154,  0.2977,  0.3099,
          0.1896,  0.5915, -0.8249,  0.5507]], device='cuda:0')
Scaled actions :  tensor([[ 0.6352,  0.9954,  0.2346, -0.3792, -0.0067,  0.1154,  0.2977,  0.3099,
          0.1896,  0.5915, -0.8249,  0.5507]], device='cuda:0')
obs :  tensor([[ 1.1671e-01,  5.1253e-01,  1.4026e+00, -1.2335e-02,  2.2990e-02,
         -9.9966e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.7228e-02,
         -5.9504e-02, -1.5621e-01,  1.8762e-01, -3.9073e-01,  1.2175e-01,
          1.3910e-02,  1.1140e-01,  1.2946e-01,  6.6821e-02, -4.1147e-01,
          1.8970e-01, -6.9945e-01, -7.6788e-02, -3.6966e-01, -1.3938e-01,
          1.0880e+00,  2.7002e-01, -4.9024e-01, -1.3824e-02,  1.1846e-03,
         -7.6085e-02,  1.2202e+00, -1.5945e-01,  6.3519e-01,  9.9537e-01,
          2.3463e-01, -3.7921e-01, -6.7406e-03,  1.1537e-01,  2.9774e-01,
          3.0986e-01,  1.8957e-01,  5.9153e-01, -8.2489e-01,  5.5

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 1.3033, -0.9144,  0.8965,  0.5515, -0.7015,  0.1159,  0.3498, -1.4693,
          1.1065, -0.5827, -0.7770, -0.4590]], device='cuda:0')
Scaled actions :  tensor([[ 1.3033, -0.9144,  0.8965,  0.5515, -0.7015,  0.1159,  0.3498, -1.4693,
          1.1065, -0.5827, -0.7770, -0.4590]], device='cuda:0')
obs :  tensor([[-0.3581,  0.1319,  0.7074,  0.0023,  0.0284, -0.9996,  1.0000,  0.0000,
          0.0000, -0.1318, -0.0517, -0.1918,  0.1340, -0.2152,  0.1271, -0.0202,
          0.1327,  0.1350,  0.0619, -0.2748,  0.2561, -0.1981,  0.1165, -0.0246,
         -0.3718,  0.6884, -0.0237,  0.0897,  0.2069,  0.0358,  0.0215,  0.2441,
          0.6079,  1.3033, -0.9144,  0.8965,  0.5515, -0.7015,  0.1159,  0.3498,
         -1.4693,  1.1065, -0.5827, -0.7770, -0.4590]], device='cuda:0')
torques: [ 200.          200.          200.         -200.         -200.
   -2.3616096   200.          178.196489     74.33275455  200.
 -200.            9.71387367]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.2251,  0.7805, -0.5134,  0.8008, -0.1481,  0.2383, -0.5729, -0.2530,
         -1.6570,  0.3517, -0.7796,  0.2358]], device='cuda:0')
Scaled actions :  tensor([[ 0.2251,  0.7805, -0.5134,  0.8008, -0.1481,  0.2383, -0.5729, -0.2530,
         -1.6570,  0.3517, -0.7796,  0.2358]], device='cuda:0')
obs :  tensor([[ 0.0146, -0.7249, -0.0795, -0.0108,  0.0346, -0.9993,  1.0000,  0.0000,
          0.0000, -0.1137, -0.0531, -0.1722,  0.1098, -0.1664,  0.1183,  0.0404,
          0.1625,  0.1732,  0.0536, -0.3335,  0.2716,  0.3329, -0.1145,  0.2058,
          0.0577, -0.1351, -0.0033,  0.4105,  0.1255,  0.3008, -0.0279, -0.7123,
         -0.3520,  0.2251,  0.7805, -0.5134,  0.8008, -0.1481,  0.2383, -0.5729,
         -0.2530, -1.6570,  0.3517, -0.7796,  0.2358]], device='cuda:0')
torques: [ 200.         -200.          200.          200.         -200.
    3.22437883  200.         -200.          200.         -200.
 -200.         -200.        ]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.7053, -0.2530, -1.0686, -1.2196,  0.0694, -0.0785, -0.3045, -0.2692,
         -0.3704, -1.4988,  0.4051,  0.2566]], device='cuda:0')
Scaled actions :  tensor([[-0.7053, -0.2530, -1.0686, -1.2196,  0.0694, -0.0785, -0.3045, -0.2692,
         -0.3704, -1.4988,  0.4051,  0.2566]], device='cuda:0')
obs :  tensor([[ 0.0101, -0.0855, -0.0862, -0.0258,  0.0339, -0.9991,  1.0000,  0.0000,
          0.0000, -0.0255, -0.0693, -0.1713,  0.1527, -0.1625,  0.1547,  0.0719,
          0.1768,  0.1984,  0.0843, -0.4842,  0.2533,  0.4792, -0.0460, -0.1626,
          0.3426,  0.0333,  0.1774, -0.0549,  0.0203, -0.0136,  0.3041, -0.6420,
         -0.0408, -0.7053, -0.2530, -1.0686, -1.2196,  0.0694, -0.0785, -0.3045,
         -0.2692, -0.3704, -1.4988,  0.4051,  0.2566]], device='cuda:0')
torques: [  37.30776115  200.         -200.          200.           -4.27617848
  -10.80785333 -200.         -200.         -200.          200.
   53.11657674    6.52220828]
データ収集:

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.1419,  0.3258,  1.0204, -1.2394,  0.0993, -0.1636,  0.2695,  0.3303,
          0.5698, -1.3598,  0.0219,  0.5803]], device='cuda:0')
Scaled actions :  tensor([[-0.1419,  0.3258,  1.0204, -1.2394,  0.0993, -0.1636,  0.2695,  0.3303,
          0.5698, -1.3598,  0.0219,  0.5803]], device='cuda:0')
obs :  tensor([[ 0.5759,  0.6131,  0.6105, -0.0134,  0.0209, -0.9997,  1.0000,  0.0000,
          0.0000,  0.0103, -0.0975, -0.2294,  0.1971, -0.0941,  0.1042,  0.0035,
          0.1473,  0.1791,  0.1364, -0.5045,  0.2540, -0.0659, -0.2164, -0.3892,
          0.1227,  0.3618, -0.3913, -0.5573, -0.2748, -0.1773,  0.2237,  0.3403,
          0.0082, -0.1419,  0.3258,  1.0204, -1.2394,  0.0993, -0.1636,  0.2695,
          0.3303,  0.5698, -1.3598,  0.0219,  0.5803]], device='cuda:0')
torques: [-200.         -116.11682836 -200.         -200.          -34.51900467
   19.97492586 -112.17266537 -200.         -200.         -200.
  200.           -2.36033607]
データ収集

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.7156,  0.4918,  1.1302, -0.2132,  0.2175,  0.6694,  0.8268,  0.7162,
          0.6891, -0.1765, -0.3397,  0.1695]], device='cuda:0')
Scaled actions :  tensor([[ 0.7156,  0.4918,  1.1302, -0.2132,  0.2175,  0.6694,  0.8268,  0.7162,
          0.6891, -0.1765, -0.3397,  0.1695]], device='cuda:0')
obs :  tensor([[ 0.0276,  0.0242,  0.5187, -0.0018,  0.0098, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0300, -0.1067, -0.2715,  0.1945, -0.0278,  0.0131, -0.0535,
          0.1226,  0.1763,  0.1390, -0.3564,  0.3388, -0.2332,  0.0830, -0.0609,
         -0.1185,  0.2773, -0.3914, -0.0601,  0.0036,  0.1292, -0.1583,  0.8164,
          0.5241,  0.7156,  0.4918,  1.1302, -0.2132,  0.2175,  0.6694,  0.8268,
          0.7162,  0.6891, -0.1765, -0.3397,  0.1695]], device='cuda:0')
torques: [   2.07409955  200.          200.         -200.          -23.85300646
   39.25706508  200.          200.          200.         -200.
  -55.31479699  -37.32538599]
データ収集

In [49]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.040, Scaled action max=1.040
Step 1/10, Total steps: 262
steps: 262
actions : tensor([[-0.7162, -0.2122,  0.2607,  0.4692, -0.6467, -0.1821, -0.7127,  0.1601,
          0.2497,  0.5160,  1.0399, -0.8755]], device='cuda:0')
target_dof_pos: tensor([[-0.3157,  0.4021, -0.8306,  1.7734, -1.1684,  0.0197,  0.2425, -0.8713,
         -0.1446,  1.1440,  1.2678, -2.0437]], device='cuda:0')
Step 1: Original action max=0.483, Scaled action max=0.483
Step 2: Original action max=1.207, Scaled action max=1.207
データ収集完了: 10 steps collected with action_scale=1.0


In [27]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [28]:
env.sim.stop()

In [29]:
env.reset()
cnt = 0

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
